In [25]:
import argparse
import glob
import os
import time
import sys
import matplotlib.pyplot as plt


import torch
import torch.nn as nn
import torch.nn.functional as F

from sparse_softmax import Sparsemax
from torch.nn import Parameter
from torch_geometric.data import Data
from torch_geometric.nn.conv import MessagePassing
from torch_geometric.nn.pool.topk_pool import topk, filter_adj
from torch_geometric.utils import softmax, dense_to_sparse, add_remaining_self_loops
from torch_scatter import scatter_add
from torch_sparse import spspmm, coalesce
from torch.utils.data import random_split
from torch_geometric.data import DataLoader
from torch_geometric.datasets import TUDataset
from torch_geometric.nn import global_mean_pool as gap, global_max_pool as gmp
from torch_geometric.nn import GCNConv


import torch.nn.utils.prune as prune
from torch.nn.utils.prune import global_unstructured, L1Unstructured

#from utils import get_model_size, get_num_parameters
Byte = 8
KiB = 1024 * Byte
MiB = 1024 * KiB
GiB = 1024 * MiB

In [2]:
import sys
sys.argv=['']
del sys

parser = argparse.ArgumentParser()

parser.add_argument('--seed', type=int, default=777, help='random seed')
parser.add_argument('--batch_size', type=int, default=512, help='batch size')
parser.add_argument('--lr', type=float, default=0.001, help='learning rate')
parser.add_argument('--weight_decay', type=float, default=0.001, help='weight decay')
parser.add_argument('--nhid', type=int, default=128, help='hidden size')
parser.add_argument('--sample_neighbor', type=bool, default=True, help='whether sample neighbors')
parser.add_argument('--sparse_attention', type=bool, default=True, help='whether use sparse attention')
parser.add_argument('--structure_learning', type=bool, default=True, help='whether perform structure learning')
parser.add_argument('--pooling_ratio', type=float, default=0.5, help='pooling ratio')
parser.add_argument('--dropout_ratio', type=float, default=0.0, help='dropout ratio')
parser.add_argument('--lamb', type=float, default=1.0, help='trade-off parameter')
parser.add_argument('--dataset', type=str, default='PROTEINS', help='DD/PROTEINS/NCI1/NCI109/Mutagenicity/ENZYMES')
parser.add_argument('--device', type=str, default='cuda:0', help='specify cuda devices')
parser.add_argument('--epochs', type=int, default=100, help='maximum number of epochs')
parser.add_argument('--patience', type=int, default=100, help='patience for early stopping')

args = parser.parse_args()
torch.manual_seed(args.seed)

In [3]:
dataset = TUDataset(os.path.join('data', args.dataset), name=args.dataset, use_node_attr=True)

args.num_classes = dataset.num_classes
args.num_features = dataset.num_features

print(args)

Namespace(seed=777, batch_size=512, lr=0.001, weight_decay=0.001, nhid=128, sample_neighbor=True, sparse_attention=True, structure_learning=True, pooling_ratio=0.5, dropout_ratio=0.0, lamb=1.0, dataset='PROTEINS', device='cuda:0', epochs=100, patience=100, num_classes=2, num_features=4)


In [4]:
num_training = int(len(dataset) * 0.8)
num_val = int(len(dataset) * 0.1)
num_test = len(dataset) - (num_training + num_val)
training_set, validation_set, test_set = random_split(dataset, [num_training, num_val, num_test])

train_loader = DataLoader(training_set, batch_size=args.batch_size, shuffle=True)
val_loader = DataLoader(validation_set, batch_size=args.batch_size, shuffle=False)
test_loader = DataLoader(test_set, batch_size=args.batch_size, shuffle=False)

C:\Users\Dell\AppData\Local\Programs\Python\Python310\lib\site-packages\torch_geometric\deprecation.py:22: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


In [10]:
def get_num_parameters(model: nn.Module, count_nonzero_only=False) -> int:
    """
    calculate the total number of parameters of model
    :param count_nonzero_only: only count nonzero weights
    """
    num_counted_elements = 0
    for param in model.parameters():
        if count_nonzero_only:
            num_counted_elements += param.count_nonzero()
        else:
            num_counted_elements += param.numel()
    return num_counted_elements


def get_model_size(model: nn.Module, data_width=32, count_nonzero_only=False) -> int:
    """
    calculate the model size in bits
    :param data_width: #bits per element
    :param count_nonzero_only: only count nonzero weights
    """
    return get_num_parameters(model, count_nonzero_only) * data_width

Byte = 8
KiB = 1024 * Byte
MiB = 1024 * KiB
GiB = 1024 * MiB

In [11]:


class TwoHopNeighborhood(object):
    def __call__(self, data):
        edge_index, edge_attr = data.edge_index, data.edge_attr
        n = data.num_nodes

        fill = 1e16
        value = edge_index.new_full((edge_index.size(1),), fill, dtype=torch.float)

        index, value = spspmm(edge_index, value, edge_index, value, n, n, n, True)

        edge_index = torch.cat([edge_index, index], dim=1)
        if edge_attr is None:
            data.edge_index, _ = coalesce(edge_index, None, n, n)
        else:
            value = value.view(-1, *[1 for _ in range(edge_attr.dim() - 1)])
            value = value.expand(-1, *list(edge_attr.size())[1:])
            edge_attr = torch.cat([edge_attr, value], dim=0)
            #, fill_value=fill
            data.edge_index, edge_attr = coalesce(edge_index, edge_attr, n, n, op='min')
            edge_attr[edge_attr >= fill] = 0
            data.edge_attr = edge_attr

        return data

    def __repr__(self):
        return '{}()'.format(self.__class__.__name__)


class GCN(MessagePassing):
    def __init__(self, in_channels, out_channels, cached=False, bias=True, **kwargs):
        super(GCN, self).__init__(aggr='add', **kwargs)

        self.in_channels = in_channels
        self.out_channels = out_channels
        self.cached = cached
        self.cached_result = None
        self.cached_num_edges = None

        self.weight = Parameter(torch.Tensor(in_channels, out_channels))
        nn.init.xavier_uniform_(self.weight.data)

        if bias:
            self.bias = Parameter(torch.Tensor(out_channels))
            nn.init.zeros_(self.bias.data)
        else:
            self.register_parameter('bias', None)

        self.reset_parameters()

    def reset_parameters(self):
        self.cached_result = None
        self.cached_num_edges = None

    @staticmethod
    def norm(edge_index, num_nodes, edge_weight, dtype=None):
        if edge_weight is None:
            edge_weight = torch.ones((edge_index.size(1),), dtype=dtype, device=edge_index.device)

        row, col = edge_index
        deg = scatter_add(edge_weight, row, dim=0, dim_size=num_nodes)
        deg_inv_sqrt = deg.pow(-0.5)
        deg_inv_sqrt[deg_inv_sqrt == float('inf')] = 0

        return edge_index, deg_inv_sqrt[row] * edge_weight * deg_inv_sqrt[col]

    def forward(self, x, edge_index, edge_weight=None):
        x = torch.matmul(x, self.weight)

        if self.cached and self.cached_result is not None:
            if edge_index.size(1) != self.cached_num_edges:
                raise RuntimeError(
                    'Cached {} number of edges, but found {}'.format(self.cached_num_edges, edge_index.size(1)))

        if not self.cached or self.cached_result is None:
            self.cached_num_edges = edge_index.size(1)
            edge_index, norm = self.norm(edge_index, x.size(0), edge_weight, x.dtype)
            self.cached_result = edge_index, norm

        edge_index, norm = self.cached_result

        return self.propagate(edge_index, x=x, norm=norm)

    def message(self, x_j, norm):
        return norm.view(-1, 1) * x_j

    def update(self, aggr_out):
        if self.bias is not None:
            aggr_out = aggr_out + self.bias
        return aggr_out

    def __repr__(self):
        return '{}({}, {})'.format(self.__class__.__name__, self.in_channels, self.out_channels)


class NodeInformationScore(MessagePassing):
    def __init__(self, improved=False, cached=False, **kwargs):
        super(NodeInformationScore, self).__init__(aggr='add', **kwargs)

        self.improved = improved
        self.cached = cached
        self.cached_result = None
        self.cached_num_edges = None

    @staticmethod
    def norm(edge_index, num_nodes, edge_weight, dtype=None):
        if edge_weight is None:
            edge_weight = torch.ones((edge_index.size(1),), dtype=dtype, device=edge_index.device)

        row, col = edge_index
        deg = scatter_add(edge_weight, row, dim=0, dim_size=num_nodes)
        deg_inv_sqrt = deg.pow(-0.5)
        deg_inv_sqrt[deg_inv_sqrt == float('inf')] = 0

        edge_index, edge_weight = add_remaining_self_loops(edge_index, edge_weight, 0, num_nodes)

        row, col = edge_index
        expand_deg = torch.zeros((edge_weight.size(0),), dtype=dtype, device=edge_index.device)
        expand_deg[-num_nodes:] = torch.ones((num_nodes,), dtype=dtype, device=edge_index.device)

        return edge_index, expand_deg - deg_inv_sqrt[row] * edge_weight * deg_inv_sqrt[col]

    def forward(self, x, edge_index, edge_weight):
        if self.cached and self.cached_result is not None:
            if edge_index.size(1) != self.cached_num_edges:
                raise RuntimeError(
                    'Cached {} number of edges, but found {}'.format(self.cached_num_edges, edge_index.size(1)))

        if not self.cached or self.cached_result is None:
            self.cached_num_edges = edge_index.size(1)
            edge_index, norm = self.norm(edge_index, x.size(0), edge_weight, x.dtype)
            self.cached_result = edge_index, norm

        edge_index, norm = self.cached_result

        return self.propagate(edge_index, x=x, norm=norm)

    def message(self, x_j, norm):
        return norm.view(-1, 1) * x_j

    def update(self, aggr_out):
        return aggr_out


class HGPSLPool(torch.nn.Module):
    def __init__(self, in_channels, ratio=0.8, sample=False, sparse=False, sl=True, lamb=1.0, negative_slop=0.2):
        super(HGPSLPool, self).__init__()
        self.in_channels = in_channels
        self.ratio = ratio
        self.sample = sample
        self.sparse = sparse
        self.sl = sl
        self.negative_slop = negative_slop
        self.lamb = lamb

        self.att = Parameter(torch.Tensor(1, self.in_channels * 2))
        nn.init.xavier_uniform_(self.att.data)
        self.sparse_attention = Sparsemax()
        self.neighbor_augment = TwoHopNeighborhood()
        self.calc_information_score = NodeInformationScore()

    def forward(self, x, edge_index, edge_attr, batch=None):
        if batch is None:
            batch = edge_index.new_zeros(x.size(0))

        x_information_score = self.calc_information_score(x, edge_index, edge_attr)
        score = torch.sum(torch.abs(x_information_score), dim=1)

        # Graph Pooling
        original_x = x
        perm = topk(score, self.ratio, batch)
        x = x[perm]
        batch = batch[perm]
        induced_edge_index, induced_edge_attr = filter_adj(edge_index, edge_attr, perm, num_nodes=score.size(0))

        # Discard structure learning layer, directly return
        if self.sl is False:
            return x, induced_edge_index, induced_edge_attr, batch

        # Structure Learning
        if self.sample:
            # A fast mode for large graphs.
            # In large graphs, learning the possible edge weights between each pair of nodes is time consuming.
            # To accelerate this process, we sample it's K-Hop neighbors for each node and then learn the
            # edge weights between them.
            k_hop = 3
            if edge_attr is None:
                edge_attr = torch.ones((edge_index.size(1),), dtype=torch.float, device=edge_index.device)

            hop_data = Data(x=original_x, edge_index=edge_index, edge_attr=edge_attr)
            for _ in range(k_hop - 1):
                hop_data = self.neighbor_augment(hop_data)
            hop_edge_index = hop_data.edge_index
            hop_edge_attr = hop_data.edge_attr
            new_edge_index, new_edge_attr = filter_adj(hop_edge_index, hop_edge_attr, perm, num_nodes=score.size(0))

            new_edge_index, new_edge_attr = add_remaining_self_loops(new_edge_index, new_edge_attr, 0, x.size(0))
            row, col = new_edge_index
            weights = (torch.cat([x[row], x[col]], dim=1) * self.att).sum(dim=-1)
            weights = F.leaky_relu(weights, self.negative_slop) + new_edge_attr * self.lamb
            adj = torch.zeros((x.size(0), x.size(0)), dtype=torch.float, device=x.device)
            adj[row, col] = weights
            new_edge_index, weights = dense_to_sparse(adj)
            row, col = new_edge_index
            if self.sparse:
                new_edge_attr = self.sparse_attention(weights, row)
            else:
                new_edge_attr = softmax(weights, row, x.size(0))
            # filter out zero weight edges
            adj[row, col] = new_edge_attr
            new_edge_index, new_edge_attr = dense_to_sparse(adj)
            # release gpu memory
            del adj
            torch.cuda.empty_cache()
        else:
            # Learning the possible edge weights between each pair of nodes in the pooled subgraph, relative slower.
            if edge_attr is None:
                induced_edge_attr = torch.ones((induced_edge_index.size(1),), dtype=x.dtype,
                                               device=induced_edge_index.device)
            num_nodes = scatter_add(batch.new_ones(x.size(0)), batch, dim=0)
            shift_cum_num_nodes = torch.cat([num_nodes.new_zeros(1), num_nodes.cumsum(dim=0)[:-1]], dim=0)
            cum_num_nodes = num_nodes.cumsum(dim=0)
            adj = torch.zeros((x.size(0), x.size(0)), dtype=torch.float, device=x.device)
            # Construct batch fully connected graph in block diagonal matirx format
            for idx_i, idx_j in zip(shift_cum_num_nodes, cum_num_nodes):
                adj[idx_i:idx_j, idx_i:idx_j] = 1.0
            new_edge_index, _ = dense_to_sparse(adj)
            row, col = new_edge_index

            weights = (torch.cat([x[row], x[col]], dim=1) * self.att).sum(dim=-1)
            weights = F.leaky_relu(weights, self.negative_slop)
            adj[row, col] = weights
            induced_row, induced_col = induced_edge_index

            adj[induced_row, induced_col] += induced_edge_attr * self.lamb
            weights = adj[row, col]
            if self.sparse:
                new_edge_attr = self.sparse_attention(weights, row)
            else:
                new_edge_attr = softmax(weights, row, x.size(0))
            # filter out zero weight edges
            adj[row, col] = new_edge_attr
            new_edge_index, new_edge_attr = dense_to_sparse(adj)
            # release gpu memory
            del adj
            torch.cuda.empty_cache()

        return x, new_edge_index, new_edge_attr, batch


In [12]:



class Model(torch.nn.Module):
    def __init__(self, args):
        super(Model, self).__init__()
        self.args = args
        self.num_features = args.num_features
        self.nhid = args.nhid
        self.num_classes = args.num_classes
        self.pooling_ratio = args.pooling_ratio
        self.dropout_ratio = args.dropout_ratio
        self.sample = args.sample_neighbor
        self.sparse = args.sparse_attention
        self.sl = args.structure_learning
        self.lamb = args.lamb

        self.conv1 = GCNConv(self.num_features, self.nhid)
        self.conv2 = GCN(self.nhid, self.nhid)
        self.conv3 = GCN(self.nhid, self.nhid)

        self.pool1 = HGPSLPool(self.nhid, self.pooling_ratio, self.sample, self.sparse, self.sl, self.lamb)
        self.pool2 = HGPSLPool(self.nhid, self.pooling_ratio, self.sample, self.sparse, self.sl, self.lamb)

        self.lin1 = torch.nn.Linear(self.nhid * 2, self.nhid)
        self.lin2 = torch.nn.Linear(self.nhid, self.nhid // 2)
        self.lin3 = torch.nn.Linear(self.nhid // 2, self.num_classes)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        edge_attr = None

        x = F.relu(self.conv1(x, edge_index, edge_attr))
        x, edge_index, edge_attr, batch = self.pool1(x, edge_index, edge_attr, batch)
        x1 = torch.cat([gmp(x, batch), gap(x, batch)], dim=1)

        x = F.relu(self.conv2(x, edge_index, edge_attr))
        x, edge_index, edge_attr, batch = self.pool2(x, edge_index, edge_attr, batch)
        x2 = torch.cat([gmp(x, batch), gap(x, batch)], dim=1)

        x = F.relu(self.conv3(x, edge_index, edge_attr))
        x3 = torch.cat([gmp(x, batch), gap(x, batch)], dim=1)

        x = F.relu(x1) + F.relu(x2) + F.relu(x3)

        x = F.relu(self.lin1(x))
        x = F.dropout(x, p=self.dropout_ratio, training=self.training)
        x = F.relu(self.lin2(x))
        x = F.dropout(x, p=self.dropout_ratio, training=self.training)
        x = F.log_softmax(self.lin3(x), dim=-1)

        return x


In [13]:
model = Model(args)
optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)

In [14]:
def train(model, train_loader):
    loss_train = 0.0
    correct = 0
    model.train()
    optimizer.zero_grad()
    for i, data in enumerate(train_loader):
            #data = data.to(args.device)
            out = model(data)
            loss = F.nll_loss(out, data.y)
            loss.backward()
            optimizer.step()
            loss_train += loss.item()
            pred = out.max(dim=1)[1]
            correct += pred.eq(data.y).sum().item()
    acc_train = correct / len(train_loader.dataset)
    
    return   loss_train,acc_train      


def compute_test(loader):
    model.eval()
    correct = 0.0
    loss_test = 0.0
    for data in loader:
        #data = data.to(args.device)
        out = model(data)
        pred = out.max(dim=1)[1]
        correct += pred.eq(data.y).sum().item()
        loss_test += F.nll_loss(out, data.y).item()
    return correct / len(loader.dataset), loss_test


In [18]:
def run(model, train_loader, callbacks = None):
    min_loss = 1e10
    patience_cnt = 0
    val_loss_values = []
    best_epoch = 0

    #model.train()
    t = time.time()
    for epoch in range(50):
        #loss_train = 0.0
        #correct = 0
        loss_train,acc_train =train(model, train_loader)
        
        acc_val, loss_val = compute_test(val_loader)
        
        if epoch % 10 == 0:
            print('Epoch: {:04d}'.format(epoch ), 'loss_train: {:.6f}'.format(loss_train),
                  'acc_train: {:.6f}'.format(acc_train), 'loss_val: {:.6f}'.format(loss_val),
                  'acc_val: {:.6f}'.format(acc_val), 'time: {:.6f}s'.format(time.time() - t))

        val_loss_values.append(loss_val)
        torch.save(model.state_dict(), '{}.pth'.format(epoch))
        if val_loss_values[-1] < min_loss:
            min_loss = val_loss_values[-1]
            best_epoch = epoch
            patience_cnt = 0
        else:
            patience_cnt += 1

        if patience_cnt == args.patience:
            break

        files = glob.glob('*.pth')
        for f in files:
            epoch_nb = int(f.split('.')[0])
            if epoch_nb < best_epoch:
                os.remove(f)
    

    print('Optimization Finished! Total time elapsed: {:.6f}'.format(time.time() - t))
    
    if callbacks is not None:
            for callback in callbacks:
                callback()
    return best_epoch

## Training

In [24]:
files = glob.glob('*.pth')
for f in files:
     #if f!='{}.pth'.format(best_model):
            os.remove(f)

print(f'Training and evaluation before pruning ')
model = Model(args)
optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
best_model =run(model, train_loader)


# Restore best model for test set
model.load_state_dict(torch.load('{}.pth'.format(best_model)))


t0=time.time()
base_model_accuracy, test_loss = compute_test(test_loader)
t1=time.time()
t_base_model=t1 - t0
###
base_model_size = get_model_size(model,count_nonzero_only=True)
num_parm_base_model=get_num_parameters(model, count_nonzero_only=True)
###   
print(f"dense model has accuracy on test set={base_model_accuracy:.2f}%")
 
print(f"dense model has size={base_model_size/MiB:.2f} MiB")
print(f"The time inference of base model is ={t_base_model}") 
print(f"The number of parametrs of base model is:{num_parm_base_model}")  

Training and evaluation before pruning 
Epoch: 0000 loss_train: 1.336775 acc_train: 0.579775 loss_val: 0.635684 acc_val: 0.621622 time: 9.813919s
Optimization Finished! Total time elapsed: 80.691209
dense model has accuracy on test set=0.73%
dense model has size=0.29 MiB
The time inference of base model is =0.5638623237609863
The number of parametrs of base model is:75423


# Let's Prune the Model and Re-Evaluate the Accuracy.
## Gobal Pruning

In [26]:
for n, m in model.named_parameters():
       
            print(n)
          

conv1.bias
conv1.lin.weight
conv2.weight
conv2.bias
conv3.weight
conv3.bias
pool1.att
pool2.att
lin1.weight
lin1.bias
lin2.weight
lin2.bias
lin3.weight
lin3.bias


In [26]:

sparsity = 0.9
parameters_to_prune = [
    (model.conv1.lin, 'weight'),
    (model.conv2, 'weight'),
    (model.conv3, 'weight'),
     (model.lin1, 'weight'),
     (model.lin2, 'weight'),
     (model.lin3, 'weight'),
]

In [27]:
def global_L1(model,parameters_to_prune, sparsity):
        global_unstructured(
        parameters_to_prune,
        pruning_method= L1Unstructured,
        amount=sparsity, )  
        
  
        for module in parameters_to_prune:
           prune.remove(module[0],'weight')



In [28]:
global_L1(model,parameters_to_prune, sparsity)

In [29]:
def get_model_sparsity(model: nn.Module) -> float:
        Sparsity=dict()
        global_zero=0
        global_nzero=0
        layyers=[]
        spars=[]
        for name, param in model.named_parameters(): 
            if 'weight' in name:
                        zero=float(torch.sum(param == 0))
                        nzero=float(param.nelement())
                        sparsity=  float(zero)/ float(nzero)
                        print( f'Sparsity in {name}: {sparsity:.3f}' )
                        layyers.append(name)
                        spars.append(sparsity)
                      

                        global_zero +=zero
                        global_nzero +=nzero
                        


        Sparsity={key: value for key, value in zip(layyers,spars)}
        global_sparsity= float(global_zero) /float(global_nzero)
        Sparsity.update({'Global sparsity':  global_sparsity})
        print("Global sparsity: {:.3f}".format(global_sparsity))
        return   Sparsity   

In [30]:
get_model_sparsity(model)

Sparsity in conv1.lin.weight: 0.432
Sparsity in conv2.weight: 0.794
Sparsity in conv3.weight: 0.818
Sparsity in lin1.weight: 0.995
Sparsity in lin2.weight: 0.929
Sparsity in lin3.weight: 0.727
Global sparsity: 0.900


{'conv1.lin.weight': 0.431640625,
 'conv2.weight': 0.79388427734375,
 'conv3.weight': 0.8177490234375,
 'lin1.weight': 0.994964599609375,
 'lin2.weight': 0.9288330078125,
 'lin3.weight': 0.7265625,
 'Global sparsity': 0.8999973106712564}

### Let's Prune the Model and Re-Evaluate the Accuracy.

In [19]:
t0=time.time()
pruned_model_accuracy, test_loss = compute_test(test_loader)
t1=time.time()
t_pruned_model=t1 - t0

###
pruned_model_size = get_model_size(model,count_nonzero_only=True)
num_parm_pruned_model=get_num_parameters(model, count_nonzero_only=True)
###   
print(f"{sparsity*100}% sparse model has accuracy on test set={pruned_model_accuracy:.2f}%")
print(f"{sparsity*100}% sparse model has size={pruned_model_size/MiB:.2f} MiB")
print(f"The time inference of {sparsity*100}% sparse model is ={t_pruned_model}") 
print(f"The number of parametrs of {sparsity*100}% sparse model is:{num_parm_pruned_model}")
print(f"{sparsity*100}% sparse model has size={pruned_model_size/MiB:.2f} MiB, "
      f"which is {base_model_size/pruned_model_size:.2f}X smaller than "
      f"the {base_model_size/MiB:.2f} MiB dense model")



90.0% sparse model has accuracy on test set=0.30%
90.0% sparse model has size=0.03 MiB
The time inference of 90.0% sparse model is =0.544121265411377
The number of parametrs of 90.0% sparse model is:8490
90.0% sparse model has size=0.03 MiB, which is 8.88X smaller than the 0.29 MiB dense model


### Let's Fine-tune the Pruned Model to Get Higher Accuracy

In [20]:
print('_______________________________________________________')
print(f'Finetuning Fine-grained Pruned Sparse Model')

import os
files = glob.glob('*.pth')
for f in files:
     #if f!='{}.pth'.format(best_model):
            os.remove(f)

#best_sparse_checkpoint = dict()
#best_sparse_accuracy = 0
best_model_pruned=run(model, train_loader, callbacks=[lambda: global_L1(model,parameters_to_prune, sparsity)])


# Restore best model for test set
#model.load_state_dict(torch.load('{}.pth'.format(best_model_pruned)))
t0=time.time()
pruned_finetune_model_accuracy, test_loss = compute_test(test_loader)
t1=time.time()
t_pruned_finetune_model=t1 - t0


###
pruned_finetune_model_size = get_model_size(model,count_nonzero_only=True)
num_parm_pruned_finetune_model=get_num_parameters(model, count_nonzero_only=True)
###   
print(f"{sparsity*100}% sparse model has accuracy on test set={pruned_finetune_model_accuracy:.2f}%")
print(f"{sparsity*100}% sparse model has size={pruned_finetune_model_size/MiB:.2f} MiB")
print(f"The time inference of {sparsity*100}% sparse model is ={t_pruned_finetune_model}") 
print(f"The number of parametrs of {sparsity*100}% sparse model is:{num_parm_pruned_finetune_model}")
print(f"{sparsity*100}% sparse model has size={pruned_finetune_model_size/MiB:.2f} MiB, "
      f"which is {base_model_size/pruned_finetune_model_size:.2f}X smaller than "
  f"the {base_model_size/MiB:.2f} MiB base model")



_______________________________________________________
Finetuning Fine-grained Pruned Sparse Model
Epoch: 0000 loss_train: 1.400482 acc_train: 0.420225 loss_val: 0.703226 acc_val: 0.378378 time: 7.422303s
Optimization Finished! Total time elapsed: 76.522870
90.0% sparse model has accuracy on test set=0.30%
90.0% sparse model has size=0.03 MiB
The time inference of 90.0% sparse model is =0.509981632232666
The number of parametrs of 90.0% sparse model is:8516
90.0% sparse model has size=0.03 MiB, which is 8.86X smaller than the 0.29 MiB base model


## Manual Measurement

In [31]:
import statistics as stat

sparsity = 0.9

Eva_final=dict()


Base_model_accuracy=[]
T_base_model=[]
Num_parm_base_model=[]
Base_model_size=[]

Pruned_model_accuracy=[]
T_pruned_model=[]
Num_parm_pruned_model=[]
Pruned_model_size=[]

Pruned_finetune_model_accuracy=[]
T_pruned_finetune_model=[]
Num_parm_pruned_finetune_model=[]
Pruned_finetune_model_size=[]


Spar_model_conv1_lin_w=[]
Spar_model_conv2_w=[]
Spar_model_conv3_w=[]
Spar_model_lin1_w=[]
Spar_model_lin2_w=[]
Spar_model_lin3_w=[]
Global_spar=[] 



In [37]:
files = glob.glob('*.pth')
for f in files:
     #if f!='{}.pth'.format(best_model):
            os.remove(f)
Eva=dict()

print(f'Training and evaluation before pruning ')
model = Model(args)
optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
best_model =run(model, train_loader)


# Restore best model for test set
model.load_state_dict(torch.load('{}.pth'.format(best_model)))

t0=time.time()
base_model_accuracy, test_loss = compute_test(test_loader)
t1=time.time()
t_base_model=t1 - t0
###
base_model_size = get_model_size(model,count_nonzero_only=True)
num_parm_base_model=get_num_parameters(model, count_nonzero_only=True)
###   
print(f"dense model has accuracy on test set={base_model_accuracy:.2f}%")
print(f"dense model has size={base_model_size/MiB:.2f} MiB")
print(f"The time inference of base model is ={t_base_model}") 
print(f"The number of parametrs of base model is:{num_parm_base_model}")     

#Update my Eva dictionary
Eva.update({'base model accuracy': base_model_accuracy,
            'time inference of base model': t_base_model,
            'number parmameters of base model': num_parm_base_model,
            'size of base model': base_model_size})


print('_______________________________________________________')
print(f'Prune the Model and Re-Evaluate the Accuracy')

parameters_to_prune = [
    (model.conv1.lin, 'weight'),
    (model.conv2, 'weight'),
    (model.conv3, 'weight'),
     (model.lin1, 'weight'),
     (model.lin2, 'weight'),
     (model.lin3, 'weight'),
]
global_L1(model,parameters_to_prune, sparsity)   
###Sparsities of layyer
spar_dict=get_model_sparsity(model)  
Eva.update(spar_dict)




t0=time.time()
pruned_model_accuracy, test_loss = compute_test(test_loader)
t1=time.time()
t_pruned_model=t1 - t0

###
pruned_model_size = get_model_size(model,count_nonzero_only=True)
num_parm_pruned_model=get_num_parameters(model, count_nonzero_only=True)
###   
print(f"{sparsity*100}% sparse model has accuracy on test set={pruned_model_accuracy:.2f}%")
print(f"{sparsity*100}% sparse model has size={pruned_model_size/MiB:.2f} MiB")
print(f"The time inference of {sparsity*100}% sparse model is ={t_pruned_model}") 
print(f"The number of parametrs of {sparsity*100}% sparse model is:{num_parm_pruned_model}")
print(f"{sparsity*100}% sparse model has size={pruned_model_size/MiB:.2f} MiB, "
      f"which is {base_model_size/pruned_model_size:.2f}X smaller than "
      f"the {base_model_size/MiB:.2f} MiB dense model")



#Update my Eva dictionary
Eva.update({'pruned model accuracy': pruned_model_accuracy,
            'time inference of pruned model': t_pruned_model,
            'number parmameters of pruned model': num_parm_pruned_model,
           'size of pruned model': pruned_model_size})

print('_______________________________________________________')
print(f'Finetuning Fine-grained Pruned Sparse Model')

import os
files = glob.glob('*.pth')
for f in files:
     #if f!='{}.pth'.format(best_model):
            os.remove(f)

#best_sparse_checkpoint = dict()
#best_sparse_accuracy = 0
best_model_pruned=run(model, train_loader, callbacks=[lambda: global_L1(model,parameters_to_prune, sparsity)])



t0=time.time()
pruned_finetune_model_accuracy, test_loss = compute_test(test_loader)
t1=time.time()
t_pruned_finetune_model=t1 - t0


###
pruned_finetune_model_size = get_model_size(model,count_nonzero_only=True)
num_parm_pruned_finetune_model=get_num_parameters(model, count_nonzero_only=True)
###   
print(f"{sparsity*100}% sparse model has accuracy on test set={pruned_finetune_model_accuracy:.2f}%")
print(f"{sparsity*100}% sparse model has size={pruned_finetune_model_size/MiB:.2f} MiB")
print(f"The time inference of {sparsity*100}% sparse model is ={t_pruned_finetune_model}") 
print(f"The number of parametrs of {sparsity*100}% sparse model is:{num_parm_pruned_finetune_model}")
print(f"{sparsity*100}% sparse model has size={pruned_finetune_model_size/MiB:.2f} MiB, "
      f"which is {base_model_size/pruned_finetune_model_size:.2f}X smaller than "
      f"the {base_model_size/MiB:.2f} MiB base model")


 #Update my Eva dictionary
Eva.update({'pruned and finetune model accuracy': pruned_finetune_model_accuracy,
            'time inference of pruned and finetune model': t_pruned_finetune_model,
            'number parmameters of pruned and finetune model': num_parm_pruned_finetune_model,
            'size of pruned and finetune model':  pruned_finetune_model_size})



Base_model_accuracy.append(Eva['base model accuracy'])
T_base_model.append(Eva['time inference of base model'])
Num_parm_base_model.append(int(Eva['number parmameters of base model']))
Base_model_size.append(int(Eva['size of base model']))

Pruned_model_accuracy.append(Eva['pruned model accuracy'])
T_pruned_model.append(Eva['time inference of pruned model'])
Num_parm_pruned_model.append(int(Eva['number parmameters of pruned model']))
Pruned_model_size.append(int(Eva['size of pruned model']))

Pruned_finetune_model_accuracy.append(Eva['pruned and finetune model accuracy'])
T_pruned_finetune_model.append(Eva['time inference of pruned and finetune model'])
Num_parm_pruned_finetune_model.append(int(Eva['number parmameters of pruned and finetune model']))
Pruned_finetune_model_size.append(int(Eva['size of pruned and finetune model']))


Spar_model_conv1_lin_w.append(Eva['conv1.lin.weight'])
Spar_model_conv2_w.append(Eva['conv2.weight'])
Spar_model_conv3_w.append(Eva['conv3.weight'])
Spar_model_lin1_w.append(Eva['lin1.weight'])
Spar_model_lin2_w.append(Eva['lin2.weight'])
Spar_model_lin3_w.append(Eva['lin3.weight'])
Global_spar.append(Eva['Global sparsity'])    


Training and evaluation before pruning 
Epoch: 0000 loss_train: 1.555337 acc_train: 0.503371 loss_val: 0.663615 acc_val: 0.621622 time: 9.474127s
Epoch: 0010 loss_train: 1.313554 acc_train: 0.656180 loss_val: 0.654860 acc_val: 0.738739 time: 101.113472s
Epoch: 0020 loss_train: 1.268284 acc_train: 0.644944 loss_val: 0.598481 acc_val: 0.720721 time: 184.307652s
Epoch: 0030 loss_train: 1.234607 acc_train: 0.664045 loss_val: 0.593901 acc_val: 0.747748 time: 262.570632s
Epoch: 0040 loss_train: 1.240285 acc_train: 0.673034 loss_val: 0.583448 acc_val: 0.747748 time: 350.171775s
Optimization Finished! Total time elapsed: 423.481959
dense model has accuracy on test set=0.81%
dense model has size=0.29 MiB
The time inference of base model is =0.5127198696136475
The number of parametrs of base model is:75424
_______________________________________________________
Prune the Model and Re-Evaluate the Accuracy
Sparsity in conv1.lin.weight: 0.529
Sparsity in conv2.weight: 0.784
Sparsity in conv3.weigh

In [38]:
Spar_model_lin3_w

[0.578125, 0.703125, 0.609375, 0.6484375, 0.6484375]

In [39]:
Eva_final=dict()
base_model_accuracy_mean = stat.mean(Base_model_accuracy)
base_model_accuracy_std =  stat.stdev(Base_model_accuracy)
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)

Eva_final.update({'base model accuracy':float(format(base_model_accuracy_mean, '.3f'))})
Eva_final.update({'Std of base model accuracy':float(format(base_model_accuracy_std, '.3f'))})
                 
t_base_model_mean =stat.mean(T_base_model)
t_base_model_std =stat.stdev(T_base_model)  
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)
Eva_final.update({'time inference of base model':float(format(t_base_model_mean, '.3f'))})
Eva_final.update({'Std of time inference of base model':float(format(t_base_model_std, '.3f'))})


num_parm_base_model_mean = stat.mean(Num_parm_base_model)
num_parm_base_model_std = stat.stdev(Num_parm_base_model)
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)
Eva_final.update({'number parmameters of base model':num_parm_base_model_mean})
Eva_final.update({'Std of number parmameters of base model':num_parm_base_model_std})

base_model_size_mean = stat.mean(Base_model_size)
base_model_size_std = stat.stdev(Base_model_size)
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)
Eva_final.update({'base_model_size':base_model_size_mean})
Eva_final.update({'Std of base_model_size':base_model_size_std})

#################################

pruned_model_accuracy_mean =stat.mean(Pruned_model_accuracy)
pruned_model_accuracy_std = stat.stdev(Pruned_model_accuracy)
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)
Eva_final.update({'pruned model accuracy':float(format(pruned_model_accuracy_mean, '.3f'))})
Eva_final.update({'Std of pruned model accuracy':float(format(pruned_model_accuracy_std, '.3f'))})
                 

t_pruned_model_mean = stat.mean(T_pruned_model)
t_pruned_model_std =stat.stdev(T_pruned_model)
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)
Eva_final.update({'time inference of pruned model':float(format(t_pruned_model_mean, '.3f'))})
Eva_final.update({'Std of time inference of pruned model':float(format(t_pruned_model_std, '.3f'))})

num_parm_pruned_model_mean = stat.mean(Num_parm_pruned_model)
num_parm_pruned_model_std = stat.stdev(Num_parm_pruned_model)
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)
Eva_final.update({'number parmameters of pruned model':num_parm_pruned_model_mean})
Eva_final.update({'Std of number parmameters of pruned model':num_parm_pruned_model_std})

pruned_model_size_mean =stat.mean( Pruned_model_size)
pruned_model_size_std = stat.stdev(Pruned_model_size)
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)
Eva_final.update({'pruned model size':pruned_model_size_mean})
Eva_final.update({'Std of pruned_model_size':pruned_model_size_std })

#################################
pruned_finetune_model_accuracy_mean =stat.mean(Pruned_finetune_model_accuracy)
pruned_finetune_model_accuracy_std = stat.stdev(Pruned_finetune_model_accuracy)
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)
Eva_final.update({'pruned finetune model accuracy':float(format(pruned_finetune_model_accuracy_mean, '.3f'))})
Eva_final.update({'Std of pruned finetune model accuracy':float(format(pruned_finetune_model_accuracy_std, '.3f'))})                 

t_pruned_finetune_model_mean =stat.mean(T_pruned_finetune_model)
t_pruned_finetune_model_std =stat.stdev(T_pruned_finetune_model)
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)
Eva_final.update({'time inference of pruned finetune model':float(format(t_pruned_finetune_model_mean,'.3f'))})
Eva_final.update({'Std of time inference of pruned finetune model':float(format(t_pruned_finetune_model_std,'.3f'))})

num_parm_pruned_finetune_model_mean =stat.mean(Num_parm_pruned_finetune_model)
num_parm_pruned_finetune_model_std = stat.stdev(Num_parm_pruned_finetune_model)
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)
Eva_final.update({'number parmameters of pruned finetune model':num_parm_pruned_finetune_model_mean})
Eva_final.update({'Std of number parmameters of pruned finetune model':num_parm_pruned_finetune_model_std })

pruned_finetune_model_size_mean = stat.mean(Pruned_finetune_model_size)
pruned_finetune_model_size_std = stat.stdev(Pruned_finetune_model_size)
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)
Eva_final.update({'pruned finetune model size':pruned_finetune_model_size_mean})
Eva_final.update({'Std of pruned finetune model size':pruned_finetune_model_size_std})

Spar_model_conv1_lin_w.append(Eva['conv1.lin.weight'])
Spar_model_conv2_w.append(Eva['conv2.weight'])
Spar_model_conv3_w.append(Eva['conv3.weight'])
Spar_model_lin1_w.append(Eva['lin1.weight'])
Spar_model_lin2_w.append(Eva['lin2.weight'])
Spar_model_lin3_w.append(Eva['lin3.weight'])




sparsity_conv1_lin_w_mean = stat.mean(Spar_model_conv1_lin_w)
sparsity_conv1_lin_w_std = stat.stdev(Spar_model_conv1_lin_w)
Eva_final.update({'Sparsity in conv1.lin.weight':float(format(sparsity_conv1_lin_w_mean,'.3f'))})
Eva_final.update({'Std of Sparsity in conv1.lin.weight':float(format(sparsity_conv1_lin_w_std,'.3f'))})

sparsity_conv2_w_mean = stat.mean(Spar_model_conv2_w)
sparsity_conv2_w_std = stat.stdev(Spar_model_conv2_w)
Eva_final.update({'Sparsity in conv2.weight':float(format(sparsity_conv2_w_mean,'.3f'))})
Eva_final.update({'Std of Sparsity in conv2.weight':float(format(sparsity_conv2_w_std,'.3f'))})

sparsity_conv3_w_mean = stat.mean(Spar_model_conv3_w)
sparsity_conv3_w_std = stat.stdev(Spar_model_conv3_w)
Eva_final.update({'Sparsity in conv3.weight':float(format(sparsity_conv3_w_mean,'.3f'))})
Eva_final.update({'Std of Sparsity in conv3.weight':float(format(sparsity_conv3_w_std,'.3f'))})



sparsity_model_lin1_w_mean = stat.mean(Spar_model_lin1_w)
sparsity_model_lin1_w_std = stat.stdev(Spar_model_lin1_w)
Eva_final.update({'Sparsity in lin1.weight':float(format(sparsity_model_lin1_w_mean,'.3f'))})
Eva_final.update({'Std of Sparsity in lin1.weight':float(format(sparsity_model_lin1_w_std,'.3f'))})


sparsity_model_lin2_w_mean = stat.mean(Spar_model_lin2_w)
sparsity_model_lin2_w_std = stat.stdev(Spar_model_lin2_w)
Eva_final.update({'Sparsity in lin2.weight':float(format(sparsity_model_lin2_w_mean,'.3f'))})
Eva_final.update({'Std of Sparsity in lin2.weight':float(format(sparsity_model_lin2_w_std,'.3f'))})

sparsity_model_lin3_w_mean = stat.mean(Spar_model_lin3_w)
sparsity_model_lin3_w_std = stat.stdev(Spar_model_lin3_w)
Eva_final.update({'Sparsity in lin3.weight':float(format(sparsity_model_lin3_w_mean,'.3f'))})
Eva_final.update({'Std of Sparsity in lin3.weight':float(format(sparsity_model_lin3_w_std,'.3f'))})


Global_sparsity_mean = stat.mean(Global_spar)
Global_sparsity_std = stat.stdev(Global_spar)
Eva_final.update({'Global sparsity':float(format(Global_sparsity_mean,'.3f'))})
Eva_final.update({'Std of Global sparsity':float(format(Global_sparsity_std,'.3f'))})


#################################


print(f"All measurement about pruning process of sparsity:{sparsity*100}% ")   
Eva_final

All measurement about pruning process of sparsity:90.0% 


{'base model accuracy': 0.807,
 'Std of base model accuracy': 0.043,
 'time inference of base model': 0.505,
 'Std of time inference of base model': 0.016,
 'number parmameters of base model': 75430.8,
 'Std of number parmameters of base model': 5.167204273105526,
 'base_model_size': 2413785.6,
 'Std of base_model_size': 165.35053673937682,
 'pruned model accuracy': 0.539,
 'Std of pruned model accuracy': 0.228,
 'time inference of pruned model': 0.505,
 'Std of time inference of pruned model': 0.011,
 'number parmameters of pruned model': 8499.8,
 'Std of number parmameters of pruned model': 5.167204273105526,
 'pruned model size': 271993.6,
 'Std of pruned_model_size': 165.35053673937682,
 'pruned finetune model accuracy': 0.53,
 'Std of pruned finetune model accuracy': 0.216,
 'time inference of pruned finetune model': 0.511,
 'Std of time inference of pruned finetune model': 0.02,
 'number parmameters of pruned finetune model': 8525.4,
 'Std of number parmameters of pruned finetune

In [40]:
Proteins_00={'base model accuracy': 0.807,
 'Std of base model accuracy': 0.043,
 'time inference of base model': 0.505,
 'Std of time inference of base model': 0.016,
 'number parmameters of base model': 75430.8,
 'Std of number parmameters of base model': 5.167204273105526,
 'base_model_size': 2413785.6,
 'Std of base_model_size': 165.35053673937682,
 'pruned model accuracy': 0.539,
 'Std of pruned model accuracy': 0.228,
 'time inference of pruned model': 0.505,
 'Std of time inference of pruned model': 0.011,
 'number parmameters of pruned model': 8499.8,
 'Std of number parmameters of pruned model': 5.167204273105526,
 'pruned model size': 271993.6,
 'Std of pruned_model_size': 165.35053673937682,
 'pruned finetune model accuracy': 0.53,
 'Std of pruned finetune model accuracy': 0.216,
 'time inference of pruned finetune model': 0.511,
 'Std of time inference of pruned finetune model': 0.02,
 'number parmameters of pruned finetune model': 8525.4,
 'Std of number parmameters of pruned finetune model': 1.51657508881031,
 'pruned finetune model size': 272812.8,
 'Std of pruned finetune model size': 48.53040284192992,
 'Sparsity in conv1.lin.weight': 0.475,
 'Std of Sparsity in conv1.lin.weight': 0.045,
 'Sparsity in conv2.weight': 0.797,
 'Std of Sparsity in conv2.weight': 0.01,
 'Sparsity in conv3.weight': 0.83,
 'Std of Sparsity in conv3.weight': 0.008,
 'Sparsity in lin1.weight': 0.99,
 'Std of Sparsity in lin1.weight': 0.006,
 'Sparsity in lin2.weight': 0.917,
 'Std of Sparsity in lin2.weight': 0.01,
 'Sparsity in lin3.weight': 0.639,
 'Std of Sparsity in lin3.weight': 0.042,
 'Global sparsity': 0.9,
 'Std of Global sparsity': 0.0}